# Build Manifest

In [2]:
import boto3, s3fs, gzip, io, os, tempfile, time
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm

BUCKET      = "echodata25"
ROOT_PREFIX = "results/echo-study/"
DST_KEY     = "results/echo-images/all_unmasked_png_paths_0_v2.clean.txt.gz"

THREADS       = 64           # tune 32-128 depending on instance CPU/network
STUDY_PAGE    = 1_000        # list 1 000 study prefixes per call
PNG_PAGE      = 1_000
FLUSH_LINES   = 100_000      # write to gzip every N lines

s3 = boto3.client("s3")

# ────────────────────────── step 1 ─ list study folders ──────────────────────
study_prefixes = []

paginator = s3.get_paginator("list_objects_v2")
pages = paginator.paginate(
    Bucket=BUCKET,
    Prefix=ROOT_PREFIX,
    Delimiter="/",               # <── get common prefixes (folder names)
    PaginationConfig={'PageSize': STUDY_PAGE},
)

for page in tqdm(pages, desc="study folders"):
    study_prefixes += [p["Prefix"] for p in page.get("CommonPrefixes", [])]

print(f"found {len(study_prefixes):,} study dirs")

# ───────────────── step 2 ─ list PNGs under each study in parallel ───────────
tmpf = tempfile.NamedTemporaryFile("wb", delete=False)
gz   = gzip.GzipFile(fileobj=tmpf, mode="wb", compresslevel=6)
buf  = []
lock = os.fsync                         # we just need a callable placeholder

def list_pngs(study_pref):
    paginator = s3.get_paginator("list_objects_v2")
    pages = paginator.paginate(
        Bucket=BUCKET,
        Prefix=study_pref + "unmasked/png/",
        PaginationConfig={'PageSize': PNG_PAGE},
    )
    for page in pages:
        for obj in page.get("Contents", []):
            yield obj["Key"]

def worker(study_pref):
    lines = []
    for key in list_pngs(study_pref):
        lines.append(f"s3://{BUCKET}/{key}\n")
    return "".join(lines)

t0 = time.time()
count = 0
bar = tqdm(total=len(study_prefixes), desc="studies processed")

with ThreadPoolExecutor(max_workers=THREADS) as pool:
    futures = {pool.submit(worker, p): p for p in study_prefixes}
    for fut in as_completed(futures):
        data = fut.result()
        if data:
            gz.write(data.encode())
            count += data.count("\n")
        bar.update(1)
        if count // FLUSH_LINES != (count - data.count("\n")) // FLUSH_LINES:
            # show key throughput every FLUSH_LINES
            elapsed = time.time() - t0
            speed = count / elapsed
            bar.set_postfix({"png": f"{count/1e6:.2f} M",
                             "speed": f"{speed:,.0f}/s"})

gz.close(); tmpf.close(); bar.close()
elapsed = time.time() - t0
print(f"\nlocal gzip complete  •  {count:,} PNGs  •  {elapsed/60:.1f} min")

# ───────────────── step 3 ─ single multipart upload ─────────────────────────
print("uploading …")
s3.upload_file(tmpf.name, BUCKET, DST_KEY)
os.unlink(tmpf.name)
print(f"✓ manifest at s3://{BUCKET}/{DST_KEY}")

study folders: 0it [00:00, ?it/s]

found 215,114 study dirs


studies processed:   0%|          | 0/215114 [00:00<?, ?it/s]


local gzip complete  •  12,109,030 PNGs  •  30.5 min
uploading …
✓ manifest at s3://echodata25/results/echo-images/all_unmasked_png_paths_0_v2.clean.txt.gz


In [3]:
import boto3, s3fs, gzip, io, os, tempfile, time
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm

BUCKET      = "echodata25"
ROOT_PREFIX = "results/echo-study-1/"
DST_KEY     = "results/echo-images/all_unmasked_png_paths_1_v2.clean.txt.gz"

THREADS       = 64           # tune 32-128 depending on instance CPU/network
STUDY_PAGE    = 1_000        # list 1 000 study prefixes per call
PNG_PAGE      = 1_000
FLUSH_LINES   = 100_000      # write to gzip every N lines

s3 = boto3.client("s3")

# ────────────────────────── step 1 ─ list study folders ──────────────────────
study_prefixes = []

paginator = s3.get_paginator("list_objects_v2")
pages = paginator.paginate(
    Bucket=BUCKET,
    Prefix=ROOT_PREFIX,
    Delimiter="/",               # <── get common prefixes (folder names)
    PaginationConfig={'PageSize': STUDY_PAGE},
)

for page in tqdm(pages, desc="study folders"):
    study_prefixes += [p["Prefix"] for p in page.get("CommonPrefixes", [])]

print(f"found {len(study_prefixes):,} study dirs")

# ───────────────── step 2 ─ list PNGs under each study in parallel ───────────
tmpf = tempfile.NamedTemporaryFile("wb", delete=False)
gz   = gzip.GzipFile(fileobj=tmpf, mode="wb", compresslevel=6)
buf  = []
lock = os.fsync                         # we just need a callable placeholder

def list_pngs(study_pref):
    paginator = s3.get_paginator("list_objects_v2")
    pages = paginator.paginate(
        Bucket=BUCKET,
        Prefix=study_pref + "unmasked/png/",
        PaginationConfig={'PageSize': PNG_PAGE},
    )
    for page in pages:
        for obj in page.get("Contents", []):
            yield obj["Key"]

def worker(study_pref):
    lines = []
    for key in list_pngs(study_pref):
        lines.append(f"s3://{BUCKET}/{key}\n")
    return "".join(lines)

t0 = time.time()
count = 0
bar = tqdm(total=len(study_prefixes), desc="studies processed")

with ThreadPoolExecutor(max_workers=THREADS) as pool:
    futures = {pool.submit(worker, p): p for p in study_prefixes}
    for fut in as_completed(futures):
        data = fut.result()
        if data:
            gz.write(data.encode())
            count += data.count("\n")
        bar.update(1)
        if count // FLUSH_LINES != (count - data.count("\n")) // FLUSH_LINES:
            # show key throughput every FLUSH_LINES
            elapsed = time.time() - t0
            speed = count / elapsed
            bar.set_postfix({"png": f"{count/1e6:.2f} M",
                             "speed": f"{speed:,.0f}/s"})

gz.close(); tmpf.close(); bar.close()
elapsed = time.time() - t0
print(f"\nlocal gzip complete  •  {count:,} PNGs  •  {elapsed/60:.1f} min")

# ───────────────── step 3 ─ single multipart upload ─────────────────────────
print("uploading …")
s3.upload_file(tmpf.name, BUCKET, DST_KEY)
os.unlink(tmpf.name)
print(f"✓ manifest at s3://{BUCKET}/{DST_KEY}")

study folders: 0it [00:00, ?it/s]

found 26,145 study dirs


studies processed:   0%|          | 0/26145 [00:00<?, ?it/s]


local gzip complete  •  1,584,665 PNGs  •  4.0 min
uploading …
✓ manifest at s3://echodata25/results/echo-images/all_unmasked_png_paths_1_v2.clean.txt.gz


In [4]:
import boto3, s3fs, gzip, io, os, tempfile, time
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm

BUCKET      = "echodata25"
ROOT_PREFIX = "results/echo-study-2/"
DST_KEY     = "results/echo-images/all_unmasked_png_paths_2_v2.clean.txt.gz"

THREADS       = 64           # tune 32-128 depending on instance CPU/network
STUDY_PAGE    = 1_000        # list 1 000 study prefixes per call
PNG_PAGE      = 1_000
FLUSH_LINES   = 100_000      # write to gzip every N lines

s3 = boto3.client("s3")

# ────────────────────────── step 1 ─ list study folders ──────────────────────
study_prefixes = []

paginator = s3.get_paginator("list_objects_v2")
pages = paginator.paginate(
    Bucket=BUCKET,
    Prefix=ROOT_PREFIX,
    Delimiter="/",               # <── get common prefixes (folder names)
    PaginationConfig={'PageSize': STUDY_PAGE},
)

for page in tqdm(pages, desc="study folders"):
    study_prefixes += [p["Prefix"] for p in page.get("CommonPrefixes", [])]

print(f"found {len(study_prefixes):,} study dirs")

# ───────────────── step 2 ─ list PNGs under each study in parallel ───────────
tmpf = tempfile.NamedTemporaryFile("wb", delete=False)
gz   = gzip.GzipFile(fileobj=tmpf, mode="wb", compresslevel=6)
buf  = []
lock = os.fsync                         # we just need a callable placeholder

def list_pngs(study_pref):
    paginator = s3.get_paginator("list_objects_v2")
    pages = paginator.paginate(
        Bucket=BUCKET,
        Prefix=study_pref + "unmasked/png/",
        PaginationConfig={'PageSize': PNG_PAGE},
    )
    for page in pages:
        for obj in page.get("Contents", []):
            yield obj["Key"]

def worker(study_pref):
    lines = []
    for key in list_pngs(study_pref):
        lines.append(f"s3://{BUCKET}/{key}\n")
    return "".join(lines)

t0 = time.time()
count = 0
bar = tqdm(total=len(study_prefixes), desc="studies processed")

with ThreadPoolExecutor(max_workers=THREADS) as pool:
    futures = {pool.submit(worker, p): p for p in study_prefixes}
    for fut in as_completed(futures):
        data = fut.result()
        if data:
            gz.write(data.encode())
            count += data.count("\n")
        bar.update(1)
        if count // FLUSH_LINES != (count - data.count("\n")) // FLUSH_LINES:
            # show key throughput every FLUSH_LINES
            elapsed = time.time() - t0
            speed = count / elapsed
            bar.set_postfix({"png": f"{count/1e6:.2f} M",
                             "speed": f"{speed:,.0f}/s"})

gz.close(); tmpf.close(); bar.close()
elapsed = time.time() - t0
print(f"\nlocal gzip complete  •  {count:,} PNGs  •  {elapsed/60:.1f} min")

# ───────────────── step 3 ─ single multipart upload ─────────────────────────
print("uploading …")
s3.upload_file(tmpf.name, BUCKET, DST_KEY)
os.unlink(tmpf.name)
print(f"✓ manifest at s3://{BUCKET}/{DST_KEY}")

study folders: 0it [00:00, ?it/s]

found 79,598 study dirs


studies processed:   0%|          | 0/79598 [00:00<?, ?it/s]


local gzip complete  •  5,059,180 PNGs  •  12.5 min
uploading …
✓ manifest at s3://echodata25/results/echo-images/all_unmasked_png_paths_2_v2.clean.txt.gz


# Misc

In [6]:
import pandas as pd
import s3fs

PREFIX = "s3://echodata25/results/es0_preds/"      # folder with all rank-csvs
fs = s3fs.S3FileSystem(anon=False)

# ①  find every preds_rank*.csv in that prefix
paths = fs.glob(PREFIX + "preds_rank*.csv")
print(f"found {len(paths)} files")

# ②  load each CSV into a list of DataFrames
dfs = [
    pd.read_csv(
        fs.open(p, "rb"),
        dtype={                                   # make sure probability cols stay float
            "quality": "float32", "salience": "float32",
            **{f"p_{v}": "float32" for v in [
                "a2c","a3c","a4c","a5c","plax","tee","exclude",
                "psax-av","psax-mv","psax-ap","psax-pm"]},
        },
    )
    for p in paths
]

# ③  concatenate and reset the index
es0_done = pd.concat(dfs, ignore_index=True)
print(es0_done.shape)

found 8 files
(5842515, 16)


In [7]:
es0_done.head()

,png_uri,mp4_uri,pred_view,quality,salience,p_a2c,p_a3c,p_a4c,p_a5c,p_plax,p_tee,p_exclude,p_psax-av,p_psax-mv,p_psax-ap,p_psax-pm
0,s3://echodata25/results/echo-study/1.2.276.0.7...,s3://echodata25/results/echo-study/1.2.276.0.7...,exclude,0.060332,0.649740,0.000045,0.000128,0.000078,0.000041,0.092224,0.000135,0.902344,0.001297,0.000072,0.002048,0.001384
1,s3://echodata25/results/echo-study/1.2.276.0.7...,s3://echodata25/results/echo-study/1.2.276.0.7...,a4c,0.083011,0.585450,0.001138,0.000075,0.800781,0.191772,0.000028,0.003607,0.002405,0.000000,0.000000,0.000000,0.000000
2,s3://echodata25/results/echo-study/1.2.276.0.7...,s3://echodata25/results/echo-study/1.2.276.0.7...,plax,0.091812,0.726177,0.000001,0.000002,0.000001,0.000000,0.998047,0.001256,0.000461,0.000000,0.000000,0.000000,0.000000
3,s3://echodata25/results/echo-study/1.2.276.0.7...,s3://echodata25/results/echo-study/1.2.276.0.7...,tee,0.045590,0.652495,0.000016,0.000002,0.000260,0.000010,0.000513,0.912598,0.086548,0.000001,0.000000,0.000000,0.000000
4,s3://echodata25/results/echo-study/1.2.276.0.7...,s3://echodata25/results/echo-study/1.2.276.0.7...,a2c,0.063823,0.601911,0.832520,0.148071,0.017914,0.000209,0.000001,0.000098,0.001346,0.000000,0.000000,0.000000,0.000000


In [8]:
import pandas as pd
import s3fs

PREFIX = "s3://echodata25/results/es1_preds6/"      # folder with all rank-csvs
fs = s3fs.S3FileSystem(anon=False)

# ①  find every preds_rank*.csv in that prefix
paths = fs.glob(PREFIX + "preds_rank*.csv")
print(f"found {len(paths)} files")

# ②  load each CSV into a list of DataFrames
dfs = [
    pd.read_csv(
        fs.open(p, "rb"),
        dtype={                                   # make sure probability cols stay float
            "quality": "float32", "salience": "float32",
            **{f"p_{v}": "float32" for v in [
                "a2c","a3c","a4c","a5c","plax","tee","exclude",
                "psax-av","psax-mv","psax-ap","psax-pm"]},
        },
    )
    for p in paths
]

# ③  concatenate and reset the index
es0_done = pd.concat(dfs, ignore_index=True)
print(es0_done.shape)

found 8 files
(770533, 16)


In [9]:
import pandas as pd
import s3fs

PREFIX = "s3://echodata25/results/es2_preds_dedup/"      # folder with all rank-csvs
fs = s3fs.S3FileSystem(anon=False)

# ①  find every preds_rank*.csv in that prefix
paths = fs.glob(PREFIX + "preds_rank*.csv")
print(f"found {len(paths)} files")

# ②  load each CSV into a list of DataFrames
dfs = [
    pd.read_csv(
        fs.open(p, "rb"),
        dtype={                                   # make sure probability cols stay float
            "quality": "float32", "salience": "float32",
            **{f"p_{v}": "float32" for v in [
                "a2c","a3c","a4c","a5c","plax","tee","exclude",
                "psax-av","psax-mv","psax-ap","psax-pm"]},
        },
    )
    for p in paths
]

# ③  concatenate and reset the index
es2_done = pd.concat(dfs, ignore_index=True)
print(es2_done.shape)

found 8 files
(1228678, 16)


# Classify All

In [ ]:
import boto3, s3fs, gzip, io, os, tempfile, time
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm

BUCKET      = "echodata25"
ROOT_PREFIX = "results/echo-study/"
DST_KEY     = "results/echo-images/all_unmasked_png_paths_0.clean.txt.gz"

THREADS       = 64           # tune 32-128 depending on instance CPU/network
STUDY_PAGE    = 1_000        # list 1 000 study prefixes per call
PNG_PAGE      = 1_000
FLUSH_LINES   = 100_000      # write to gzip every N lines

s3 = boto3.client("s3")

# ────────────────────────── step 1 ─ list study folders ──────────────────────
study_prefixes = []

paginator = s3.get_paginator("list_objects_v2")
pages = paginator.paginate(
    Bucket=BUCKET,
    Prefix=ROOT_PREFIX,
    Delimiter="/",               # <── get common prefixes (folder names)
    PaginationConfig={'PageSize': STUDY_PAGE},
)

for page in tqdm(pages, desc="study folders"):
    study_prefixes += [p["Prefix"] for p in page.get("CommonPrefixes", [])]

print(f"found {len(study_prefixes):,} study dirs")

# ───────────────── step 2 ─ list PNGs under each study in parallel ───────────
tmpf = tempfile.NamedTemporaryFile("wb", delete=False)
gz   = gzip.GzipFile(fileobj=tmpf, mode="wb", compresslevel=6)
buf  = []
lock = os.fsync                         # we just need a callable placeholder

def list_pngs(study_pref):
    paginator = s3.get_paginator("list_objects_v2")
    pages = paginator.paginate(
        Bucket=BUCKET,
        Prefix=study_pref + "unmasked/png/",
        PaginationConfig={'PageSize': PNG_PAGE},
    )
    for page in pages:
        for obj in page.get("Contents", []):
            yield obj["Key"]

def worker(study_pref):
    lines = []
    for key in list_pngs(study_pref):
        lines.append(f"s3://{BUCKET}/{key}\n")
    return "".join(lines)

t0 = time.time()
count = 0
bar = tqdm(total=len(study_prefixes), desc="studies processed")

with ThreadPoolExecutor(max_workers=THREADS) as pool:
    futures = {pool.submit(worker, p): p for p in study_prefixes}
    for fut in as_completed(futures):
        data = fut.result()
        if data:
            gz.write(data.encode())
            count += data.count("\n")
        bar.update(1)
        if count // FLUSH_LINES != (count - data.count("\n")) // FLUSH_LINES:
            # show key throughput every FLUSH_LINES
            elapsed = time.time() - t0
            speed = count / elapsed
            bar.set_postfix({"png": f"{count/1e6:.2f} M",
                             "speed": f"{speed:,.0f}/s"})

gz.close(); tmpf.close(); bar.close()
elapsed = time.time() - t0
print(f"\nlocal gzip complete  •  {count:,} PNGs  •  {elapsed/60:.1f} min")

# ───────────────── step 3 ─ single multipart upload ─────────────────────────
print("uploading …")
s3.upload_file(tmpf.name, BUCKET, DST_KEY)
os.unlink(tmpf.name)
print(f"✓ manifest at s3://{BUCKET}/{DST_KEY}")


In [10]:
%%writefile batch_classify.py
#!/usr/bin/env python3
# ============================================================================
#  batch_classify.py · 2025‑05‑14 (refreshed)
#  • multi‑GPU · FP16 · auto batch‑halve on OOM / IndexMath overflow
#  • strict checkpoint check ✔︎ + per‑rank skip logs + debug limit/dry‑run
#  • *NEW* crisp tqdm bars — one per rank, live ETA & throughput
# ============================================================================

"""Key UI changes
────────────────────────────────────────────────────────────────────────────────
* Each CUDA rank gets its own tqdm line (position=rank) so bars never clash.
* We pre‑count how many PNGs this rank will process. With a known total tqdm
  can compute ETA. The manifest scan adds <1 s even for multi‑million files.
* Progress bar shows imgs/s and ETA using a custom format string.
* Bar refreshes every second (mininterval).
"""

import argparse, csv, gzip, io, itertools, logging, os, re, time
from collections import Counter
from math import tanh
from typing import Iterable, Tuple, Optional

import boto3, s3fs
from botocore.config import Config as BotoCfg
from botocore.exceptions import ClientError, IncompleteReadError
from pathlib import Path
import cv2, numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
import torch.distributed as dist
from PIL import Image
from torchvision import models, transforms
from tqdm.auto import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed

# ───────────────────────────── constants ──────────────────────────────
VIEW = [
    "a2c","a3c","a4c","a5c","plax","tee","exclude",
    "psax-av","psax-mv","psax-ap","psax-pm"
]

PNG_ROW_RE = re.compile(
    r"""^s3://[^/]+/
        (?P<key>
            results/echo-study(?:-[12])?/        # echo-study/, echo-study-1/2/
            .+?/unmasked/png/
            .+\.png$
        )""",
    re.IGNORECASE | re.VERBOSE,
)

tf = transforms.Compose([
    transforms.Resize(224), transforms.CenterCrop(224), transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])

# ───────────────────── placeholders lifted to module scope ────────────
ctr: Counter = Counter()

def record_skip(kind: str, msg: str):
    ...  # rebound in main()

# ───────────────────────────── helpers ────────────────────────────────

def quality_score(img_bgr: np.ndarray) -> float:
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    sharp = cv2.Laplacian(gray, cv2.CV_64F).var()
    bright = gray.mean() / 255.0
    return tanh(0.004 * sharp) * bright

def png_to_mp4(png_key: str) -> str:
    return png_key.replace("/unmasked/png/", "/")[:-4] + ".mp4"

def chunks(it: Iterable, n: int):
    while True:
        batch = list(itertools.islice(it, n))
        if not batch:
            break
        yield batch

def retry_open(fs: s3fs.S3FileSystem, path: str, tries: int = 3):
    delay = 1.0
    for attempt in range(tries):
        try:
            return fs.open(path, "rb")
        except ClientError as e:
            if attempt == tries - 1:
                raise
            if e.response["Error"]["Code"] in ("500", "503", "InternalError"):
                time.sleep(delay)
                delay *= 1.5
            else:
                raise

# ───────────────────────────── model ──────────────────────────────────
class Net(nn.Module):
    def __init__(self):
        super().__init__()
        base = models.efficientnet_b2(weights=None)
        base.classifier = nn.Identity()
        self.b = base
        f = 1408
        self.vb = nn.Linear(f, 2)
        self.vo = nn.Linear(f, 7)
        self.vs = nn.Linear(f, 4)

    def forward(self, x):
        f = self.b(x)
        pb, po, ps = (F.softmax(h(f), 1) for h in (self.vb, self.vo, self.vs))
        out = x.new_zeros(x.size(0), 11)
        out[:, :7] = pb[:, :1] * po
        out[:, 7:] = pb[:, 1:] * ps
        return out

# ─────────────────────── manifest utilities ──────────────────────────

def open_body(s3, uri: str):
    """Download manifest once per rank → /tmp/rankX.manifest.gz, then open."""
    bucket, key = uri[5:].split("/", 1)
    local = Path(f"/tmp/manifest_rank{os.getenv('LOCAL_RANK')}.gz")
    if not local.exists():
        logging.info("rank%s downloading manifest → %s", os.getenv("LOCAL_RANK"), local)
        with s3fs.S3FileSystem(anon=False).open(f"s3://{bucket}/{key}", "rb") as src, local.open("wb") as dst:
            for chunk in iter(lambda: src.read(8 << 20), b""):
                dst.write(chunk)
    fh = local.open("rb")
    return gzip.GzipFile(fileobj=fh) if key.endswith(".gz") else fh


def iter_manifest(s3, uri: str, world: int, rank: int, limit: Optional[int]):
    seen = 0
    for idx, raw in enumerate(open_body(s3, uri)):
        if idx % world != rank:
            continue
        if limit is not None and seen >= limit:
            break
        line = raw.strip().decode()
        m = PNG_ROW_RE.match(line)
        if m:
            yield m.group("key")
            seen += 1
        else:
            ctr["regex"] += 1
            record_skip("REGEX", line)

# ───────────────────────────── main ──────────────────────────────────

def count_samples(s3, uri: str, world: int, rank: int, limit: Optional[int]) -> int:
    """Fast pass over manifest to know how many PNGs this rank owns."""
    n = 0
    for _ in iter_manifest(s3, uri, world, rank, limit):
        n += 1
    return n


def main(a):
    log_level = os.getenv("LOG_LEVEL", "INFO").upper()
    logging.basicConfig(
        format="%(asctime)s %(levelname)s │ %(message)s",
        datefmt="%H:%M:%S",
        level=getattr(logging, log_level, logging.INFO),
        force=True,
    )

    dist.init_process_group("nccl")
    rank = int(os.environ["LOCAL_RANK"])
    world = dist.get_world_size()
    dev_id = rank % torch.cuda.device_count()
    torch.cuda.set_device(dev_id)
    device = torch.device(f"cuda:{dev_id}")

    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.benchmark = True

    MAX_WORKERS = int(os.getenv("MAX_WORKERS", "256"))
    s3 = boto3.client("s3", config=BotoCfg(max_pool_connections=MAX_WORKERS))
    fs = s3fs.S3FileSystem(
        anon=False,
        default_block_size=8 << 20,
        default_fill_cache=False,
        config_kwargs={"max_pool_connections": MAX_WORKERS},
    )

    # ───── per-rank skip file ─────
    skip_path = f"/opt/ml/processing/output/skip_rank{rank}.txt.gz"
    skip_fh = gzip.open(skip_path, "wt")

    def _record(kind: str, msg: str):
        skip_fh.write(f"{kind}\t{msg}\n")

    globals()["record_skip"] = _record  # make visible globally

    # ───── dry-run? just count regex matches, exit early ─────
    if a.dry_run:
        for _ in iter_manifest(s3, a.manifest_s3, world, rank, a.limit):
            pass
        skip_fh.close()
        logging.info("DRY-RUN finished – regex %d", ctr["regex"])
        return

    # ───── progress bar prep  ─────
    total_imgs = count_samples(s3, a.manifest_s3, world, rank, a.limit)
    logging.info("rank%d will process ~%s imgs", rank, f"{total_imgs:,d}" if total_imgs else "?")

    bar = tqdm(
        total=total_imgs or None,
        desc=f"rank {rank}",
        unit="img",
        position=rank,
        dynamic_ncols=True,
        smoothing=0.1,
        mininterval=1.0,
        bar_format="{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed} < {remaining}, {rate_fmt}]",
    )

    # ───── model ─────
    net = Net().to(device).eval()
    with fs.open(a.model_s3, "rb") as f:
        state = torch.load(io.BytesIO(f.read()), map_location="cpu")
    ck = net.load_state_dict(state, strict=False)
    if ck.missing_keys or ck.unexpected_keys:
        logging.warning("‼️ checkpoint mismatch – %d missing, %d unexpected", len(ck.missing_keys), len(ck.unexpected_keys))
    else:
        logging.info("✅ checkpoint keys match perfectly")
    net.half()

    # ───── output CSV ─────
    os.makedirs("/opt/ml/processing/output", exist_ok=True)
    csv_path = f"/opt/ml/processing/output/preds_rank{rank}.csv"
    header = ["png_uri", "mp4_uri", "pred_view", "quality", "salience"] + [f"p_{v}" for v in VIEW]

    key_iter = iter_manifest(s3, a.manifest_s3, world, rank, a.limit)
    processed = 0
    t0 = time.time()

    pool = ThreadPoolExecutor(max_workers=MAX_WORKERS)

    def load_one(k: str) -> Tuple[str, torch.Tensor, float, bool]:
        try:
            with retry_open(fs, f"{a.bucket}/{k}") as f:
                arr = np.frombuffer(f.read(), np.uint8)
            img = cv2.imdecode(arr, cv2.IMREAD_COLOR)
            if img is None:
                raise ValueError("cv2.imdecode returned None")
            q = quality_score(img)
            ten = tf(Image.fromarray(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))).half()
            return k, ten, q, False
        except ClientError as ce:
            ctr["open"] += 1
            record_skip("OPEN", f"{k}\t{ce}")
        except Exception as exc:
            ctr["decode"] += 1
            record_skip("DECODE", f"{k}\t{exc}")
            logging.debug("DECODE‑fail %s — %s", k, exc)
        return k, None, None, True

    def safe_infer(b: torch.Tensor) -> np.ndarray:
        cur = b
        while True:
            try:
                with torch.cuda.amp.autocast(), torch.no_grad():
                    return net(cur).cpu().numpy()
            except (torch.cuda.OutOfMemoryError, RuntimeError) as exc:
                if ("indexmath" not in str(exc).lower() and not isinstance(exc, torch.cuda.OutOfMemoryError)):
                    raise
                torch.cuda.empty_cache()
                cur = cur[: max(128, cur.size(0) // 2)]

    with open(csv_path, "w", newline="") as fh:
        w = csv.writer(fh)
        w.writerow(header)

        for keys in chunks(key_iter, a.batch_size):
            futs = [pool.submit(load_one, k) for k in keys]
            tens, qs, oks = [], [], []
            for fut in as_completed(futs):
                k, ten, q, bad = fut.result()
                if bad:
                    continue
                tens.append(ten)
                qs.append(q)
                oks.append(k)

            if not oks:
                continue

            probs = safe_infer(torch.stack(tens).to(device, non_blocking=True))
            for k, q, p in zip(oks, qs, probs):
                sal = 0.7 * p.max() + 0.3 * q
                w.writerow([
                    f"s3://{a.bucket}/{k}",
                    f"s3://{a.bucket}/{png_to_mp4(k)}",
                    VIEW[int(p.argmax())],
                    round(q, 6),
                    round(sal, 6),
                    *map(lambda x: round(float(x), 6), p),
                ])

            processed += len(oks)
            bar.update(len(oks))

    bar.close()
    skip_fh.close()
    elapsed = time.time() - t0
    logging.info(
        "✓ finished — %.1f min | %s OK | drops %s",
        elapsed / 60,
        f"{processed:,d}",
        ", ".join(f"{k}:{v}" for k, v in ctr.items()) or "none",
    )

# ────────────────────────── CLI wiring ────────────────────────────────
if __name__ == "__main__":
    P = argparse.ArgumentParser()
    P.add_argument("--bucket", required=True)
    P.add_argument("--manifest_s3", required=True)
    P.add_argument("--model_s3", required=True)
    P.add_argument("--batch_size", type=int, default=2048)
    P.add_argument("--limit", type=int, default=None, help="debug‑only: per‑rank cap on #pngs")
    P.add_argument("--dry_run", action="store_true", help="only parse manifest + regex stats (no decoding/infer)")
    main(P.parse_args())

Writing batch_classify.py


In [ ]:
!sudo apt-get update -y
!sudo apt-get install -y docker.io
!sudo curl -L "https://github.com/docker/compose/releases/latest/download/docker-compose-$(uname -s)-$(uname -m)" -o /usr/local/bin/docker-compose
!sudo chmod +x /usr/local/bin/docker-compose
!sudo service docker start
!sudo usermod -a -G docker sagemaker-user

In [7]:
import os, time
import sagemaker
from sagemaker.pytorch import PyTorchProcessor
from sagemaker.processing import ProcessingOutput

session = sagemaker.Session()          # Studio default session
role = sagemaker.get_execution_role()  # still fine in Studio

LOCAL = True                           # flip to False to go back to remote

instance_type = "local_gpu" if LOCAL else "ml.g5.48xlarge"

# torchrun goes in `command` only if the container's entrypoint is honored.
# PyTorchProcessor uses torchrun automatically when you pass `distributed` args,
# so just call your script normally and let torchrun be handled by `command`.
processor = PyTorchProcessor(
    framework_version="2.1",
    py_version="py310",
    role=role,
    instance_type=instance_type,
    instance_count=1,
    volume_size_in_gb=100 if not LOCAL else None,   # ignored in local mode
    max_runtime_in_seconds=6*60*60 if not LOCAL else None,
    command=["torchrun", "--nproc_per_node", "8"],  # works in local mode too
    env={
        "MAX_WORKERS": "256",
        "SM_NUM_GPUS": os.environ.get("SM_NUM_GPUS", "1"),
        # "LOG_LEVEL": "DEBUG"
    },
    sagemaker_session=session,
)

outputs = [
    ProcessingOutput(
        source="/opt/ml/processing/output",
        destination=("s3://echodata25/results/es0_preds_dedup"
                     if not LOCAL else f"file://{os.getcwd()}/es0_preds_dedup"),
        output_name="preds"
    )
]

processor.run(
    code="batch_classify.py",
    arguments=[
        "--bucket", "echodata25",
        "--manifest_s3", "s3://echodata25/results/echo-images/all_unmasked_png_paths_0.clean.dedup.txt.gz",
        "--model_s3", "s3://echodata25/results/models/view_classifier/best_f1_84.pt",
        "--batch_size", "2048",
        # "--limit", "5000"
    ],
    outputs=outputs,
    job_name=(f"view-classify-{int(time.time())}"
              if not LOCAL else None),   # local mode ignores job_name
    wait=True,
    logs=True
)

print("🚀 submitted — check the local docker logs (shown here) for per-rank counters.")


INFO:sagemaker.image_uris:image_uri is not presented, retrieving image_uri based on instance_type, framework etc.
INFO:sagemaker.processing:Uploaded None to s3://sagemaker-us-west-2-495467399120/pytorch-2025-07-22-04-51-43-679/source/sourcedir.tar.gz
INFO:sagemaker.processing:runproc.sh uploaded to s3://sagemaker-us-west-2-495467399120/pytorch-2025-07-22-04-51-43-679/source/runproc.sh
INFO:sagemaker:Creating processing-job with name pytorch-2025-07-22-04-51-43-679
INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional features.
To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config. For more information, refer to https://sagemaker.readthedocs.io/en/stable/overview.html#configuring-and-using-defaults-with-the-sagemaker-python-sdk.
INFO:sagemaker.local.image:'Docker Compose' is not installed. Proceeding to check for 'docker-compose' 

╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:42                                                                                   │
│                                                                                                  │
│   39 │   )                                                                                       │
│   40 ]                                                                                           │
│   41                                                                                             │
│ ❱ 42 processor.run(                                                                              │
│   43 │   code="batch_classify.py",                                                               │
│   44 │   arguments=[                                                                             │
│   45 │   │   "--bucket", "echodata25",                                                           │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/sagemaker/workflow/pipeline_context.py:346 in wrapper    │
│                                                                                                  │
│   343 │   │   │                                                                                  │
│   344 │   │   │   return _StepArguments(retrieve_caller_name(self_instance), run_func, *args,    │
│   345 │   │                                                                                      │
│ ❱ 346 │   │   return run_func(*args, **kwargs)                                                   │
│   347 │                                                                                          │
│   348 │   return wrapper                                                                         │
│   349                                                                                            │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/sagemaker/processing.py:1780 in run                      │
│                                                                                                  │
│   1777 │   │   )                                                                                 │
│   1778 │   │                                                                                     │
│   1779 │   │   # Submit a processing job.                                                        │
│ ❱ 1780 │   │   return super().run(                                                               │
│   1781 │   │   │   code=s3_runproc_sh,                                                           │
│   1782 │   │   │   inputs=inputs,                                                                │
│   1783 │   │   │   outputs=outputs,                                                              │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/sagemaker/workflow/pipeline_context.py:346 in wrapper    │
│                                                                                                  │
│   343 │   │   │                                                                                  │
│   344 │   │   │   return _StepArguments(retrieve_caller_name(self_instance), run_func, *args,    │
│   345 │   │                                                                                      │
│ ❱ 346 │   │   return run_func(*args, **kwargs)                                                   │
│   347 │                                                                                          │
│   348 │   return wrapper                                                                         │
│   349                                                      